<div style="background-color: RGB(0,114,200);">
<h1 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">PROJET 11 — DATA ANALYST</h1>
<h2 style="margin: auto; padding: 10px 0; color:#fff; text-align: center">Produisez une étude de marché avec Python</h2>
</div>


## Sommaire

1. [Import des librairies](#import)
2. [Importation et contrôle des données](#import-donnees)
3. [Construction des variables](#variables)
4. [Nettoyage du bloc FAO et valeurs manquantes](#nettoyage)
5. [Fusion des sources (PIB et stabilité politique)](#fusion)
6. [Couverture de la population mondiale](#couverture)
7. [Enregistrement du jeu de données](#save)


<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="import"><h2 style="margin: auto; padding: 20px; color:#fff;">1. Import des librairies</h2></a>
</div>


In [119]:
#Importation des librairies
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
from matplotlib.collections import LineCollection
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn import datasets

In [120]:
# country_converter : librairie qui convertit les noms de pays en codes ISO,
# dans plusieurs langues (dont le français). On l'installe une seule fois.
!pip install country_converter

<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="import-donnees"><h2 style="margin: auto; padding: 20px; color:#fff;">2. Importation et contrôle des données</h2></a>
</div>


In [121]:
# Charger le fichier DisponibiliteAlimentaire_2017.csv
dispo = pd.read_csv("DisponibiliteAlimentaire_2017.csv")
# Aperçu des 100 premières lignes du DataFrame
dispo.head(100)

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,4281.0,S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,2302.0,S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.0,S,Données standardisées
3,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5911,Exportations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,0.0,S,Données standardisées
4,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5301,Disponibilité intérieure,2511,Blé et produits,2017,2017,Milliers de tonnes,6701.0,S,Données standardisées
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,674,Disponibilité de protéines en quantité (g/pers...,2520,"Céréales, Autres",2017,2017,g/personne/jour,0.0,Fc,Donnée calculée
96,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,684,Disponibilité de matière grasse en quantité (g...,2520,"Céréales, Autres",2017,2017,g/personne/jour,0.0,Fc,Donnée calculée
97,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2531,Pommes de Terre et produits,2017,2017,Milliers de tonnes,513.0,S,Données standardisées
98,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2531,Pommes de Terre et produits,2017,2017,Milliers de tonnes,230.0,S,Données standardisées


In [122]:
dispo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176600 entries, 0 to 176599
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Code Domaine            176600 non-null  object 
 1   Domaine                 176600 non-null  object 
 2   Code zone               176600 non-null  int64  
 3   Zone                    176600 non-null  object 
 4   Code Élément            176600 non-null  int64  
 5   Élément                 176600 non-null  object 
 6   Code Produit            176600 non-null  int64  
 7   Produit                 176600 non-null  object 
 8   Code année              176600 non-null  int64  
 9   Année                   176600 non-null  int64  
 10  Unité                   176600 non-null  object 
 11  Valeur                  176600 non-null  float64
 12  Symbole                 176600 non-null  object 
 13  Description du Symbole  176600 non-null  object 
dtypes: float64(1), int64

In [123]:

# Sélectionner les colonnes nécessaires à notre analyse 
colonnes_a_garder = ["Code zone", "Zone", "Élément", "Produit", "Unité", "Valeur"]
dispo = dispo[colonnes_a_garder]
dispo.head()

,Code zone,Zone,Élément,Produit,Unité,Valeur
0,2,Afghanistan,Production,Blé et produits,Milliers de tonnes,4281.0
1,2,Afghanistan,Importations - Quantité,Blé et produits,Milliers de tonnes,2302.0
2,2,Afghanistan,Variation de stock,Blé et produits,Milliers de tonnes,-119.0
3,2,Afghanistan,Exportations - Quantité,Blé et produits,Milliers de tonnes,0.0
4,2,Afghanistan,Disponibilité intérieure,Blé et produits,Milliers de tonnes,6701.0


In [124]:
dispo["Élément"].unique()

array(['Production', 'Importations - Quantité', 'Variation de stock',
       'Exportations - Quantité', 'Disponibilité intérieure',
       'Aliments pour animaux', 'Semences', 'Pertes', 'Résidus',
       'Nourriture',
       'Disponibilité alimentaire en quantité (kg/personne/an)',
       'Disponibilité alimentaire (Kcal/personne/jour)',
       'Disponibilité de protéines en quantité (g/personne/jour)',
       'Disponibilité de matière grasse en quantité (g/personne/jour)',
       'Traitement', 'Autres utilisations (non alimentaire)',
       'Alimentation pour touristes'], dtype=object)

In [125]:
dispo["Zone"].nunique()

174

In [126]:
dispo.isnull().sum()

Code zone    0
Zone         0
Élément      0
Produit      0
Unité        0
Valeur       0
dtype: int64

In [127]:
#Vérification des doublons
dispo.duplicated().sum()

np.int64(0)

Fichier Population

In [128]:
pop = pd.read_csv("Population_2000_2018.csv")
pop_2017 = pop[pop["Année"] == 2017]
pop_2017.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
17,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,36296.113,X,Sources internationales sûres,NaN
36,OA,Séries temporelles annuelles,202,Afrique du Sud,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,57009.756,X,Sources internationales sûres,NaN
55,OA,Séries temporelles annuelles,3,Albanie,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,2884.169,X,Sources internationales sûres,NaN
74,OA,Séries temporelles annuelles,4,Algérie,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,41389.189,X,Sources internationales sûres,NaN
93,OA,Séries temporelles annuelles,79,Allemagne,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,82658.409,X,Sources internationales sûres,NaN


In [129]:
# On garde seulement les 3 colonnes utiles
pop_2017 = pop_2017[["Code zone", "Zone", "Valeur"]]

# On renomme "Valeur" en "Population_2017" pour que ce soit clair après la fusion
pop_2017 = pop_2017.rename(columns={"Valeur": "Population_2017"})

# Vérification
pop_2017.head()

,Code zone,Zone,Population_2017
17,2,Afghanistan,36296.113
36,202,Afrique du Sud,57009.756
55,3,Albanie,2884.169
74,4,Algérie,41389.189
93,79,Allemagne,82658.409


In [130]:
# Valeurs manquantes : y a-t-il des trous dans la population 2017 ?
print("Valeurs manquantes par colonne :")
print(pop_2017.isnull().sum())

print()  # ligne vide pour aérer l'affichage

# Doublons : un pays apparaît-il deux fois ?
# On vérifie sur la colonne "Zone" (chaque pays ne doit exister qu'une fois en 2017)
print("Nombre de pays en double :")
print(pop_2017.duplicated(subset=["Zone"]).sum())

print()

# Combien de pays au total dans la population 2017 ?
print("Nombre de pays :")
print(pop_2017["Zone"].nunique())

Valeurs manquantes par colonne :
Code zone          0
Zone               0
Population_2017    0
dtype: int64

Nombre de pays en double :
0

Nombre de pays :
236


Fichier PIB

In [131]:
# Le fichier Banque mondiale a 4 lignes de "blabla" en haut (titre, date...).
# skiprows=4 dit à pandas de les ignorer pour bien lire les vraies colonnes.
pib = pd.read_csv("API_NY.GDP.PCAP.CD_DS2_en_csv_v2_465995.csv", skiprows=4)

# On garde seulement le nom du pays et la colonne de l'année 2017.
pib = pib[["Country Name", "2017"]]

# On renomme pour cohérence avec nos autres fichiers :
# - "Country Name" = "Zone"
# - "2017" = "PIB_par_hab_2017"
pib = pib.rename(columns={"Country Name": "Zone", "2017": "PIB_par_hab_2017"})

# Vérification
pib.head()

,Zone,PIB_par_hab_2017
0,Aruba,28440.041688
1,Africa Eastern and Southern,1528.104224
2,Afghanistan,525.469771
3,Africa Western and Central,1574.230564
4,Angola,2790.718869


Fichier Stabilité Politique

In [132]:
# Le fichier DataBank a des lignes de blabla à la FIN (pas au début cette fois).
# On charge normalement, puis on nettoie.
stab = pd.read_csv("f09c2b69-99f0-4702-ad1f-43a99cb25d78_Data.csv")

# On garde le nom du pays et la colonne de l'année.
stab = stab[["Country Name", "2017 [YR2017]"]]

# On renomme pour cohérence avec nos autres fichiers
stab = stab.rename(columns={
    "Country Name": "Zone",
    "2017 [YR2017]": "Stabilite_politique_2017"
})

# Gérer les lignes non imortantes
# Ces lignes ont un "Zone" vide (NaN) ou du texte comme "Data from database...".
# On convertit la colonne stabilité en nombre : tout ce qui n'est pas un nombre
# (le blabla) deviendra NaN, qu'on supprimera ensuite.
stab["Stabilite_politique_2017"] = pd.to_numeric(
    stab["Stabilite_politique_2017"], errors="coerce"
)

# On supprime les lignes où la stabilité est manquante (le blabla + pays sans donnée)
stab = stab.dropna(subset=["Stabilite_politique_2017"])

# Vérification
print("Dimensions :", stab.shape)
stab.head()

Dimensions : (212, 2)


,Zone,Stabilite_politique_2017
0,Afghanistan,-2.613931
1,Albania,0.163830
2,Algeria,-0.929505
3,American Samoa,1.255594
4,Andorra,1.325091


<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="variables"><h2 style="margin: auto; padding: 20px; color:#fff;">3. Construction des variables</h2></a>
</div>


Transformation les données de long à large

Première variable : protéines totales par habitant

In [133]:
# Objectif : obtenir pour chaque pays, le total de protéines (g/personne/jour),
# en additionnant la valeur "protéines" de tous les produits.

# Étape 1 : on garde seulement les lignes de l'élément "protéines"
proteines = dispo[dispo["Élément"] == "Disponibilité de protéines en quantité (g/personne/jour)"]

# Étape 2 : on regroupe par pays et on additionne
# - groupby("Zone") : "fais un paquet par pays"
# - ["Valeur"].sum() : "dans chaque paquet, additionne la colonne Valeur"
proteines_par_pays = proteines.groupby("Zone")["Valeur"].sum()

# On regarde les 5 premiers pays
proteines_par_pays.head()

Zone
Afghanistan        54.09
Afrique du Sud     83.36
Albanie           119.50
Algérie            92.85
Allemagne         104.07
Name: Valeur, dtype: float64

Disponibilité alimentaire totale en kcal

In [134]:
# Objectif : total de kcal par habitant pour chaque pays (somme de tous les produits)

# Étape 1 : on garde seulement les lignes "kcal"
kcal = dispo[dispo["Élément"] == "Disponibilité alimentaire (Kcal/personne/jour)"]

# Étape 2 : on regroupe par pays et on additionne
kcal_par_pays = kcal.groupby("Zone")["Valeur"].sum()

# Vérification
kcal_par_pays.head()

Zone
Afghanistan       1997.0
Afrique du Sud    2987.0
Albanie           3400.0
Algérie           3345.0
Allemagne         3559.0
Name: Valeur, dtype: float64

Taux de dépendance aux importations

In [135]:
# Objectif : pour chaque pays, calculer Importations / Disponibilité intérieure

# Étape 1 : total des IMPORTATIONS par pays (somme de tous les produits)
imports = dispo[dispo["Élément"] == "Importations - Quantité"]
imports_par_pays = imports.groupby("Zone")["Valeur"].sum()

# Étape 2 : total de la DISPONIBILITÉ INTÉRIEURE par pays
dispo_int = dispo[dispo["Élément"] == "Disponibilité intérieure"]
dispo_int_par_pays = dispo_int.groupby("Zone")["Valeur"].sum()

# Étape 3 : on divise les deux (pays par pays automatiquement, grâce à l'index Zone)
# Le résultat est une proportion (0.30 = 30% de dépendance)
taux_dependance = imports_par_pays / dispo_int_par_pays

# Vérification
taux_dependance.head()

Zone
Afghanistan       0.300879
Afrique du Sud    0.134949
Albanie           0.181185
Algérie           0.434652
Allemagne         0.380373
Name: Valeur, dtype: float64

Disponibilité de viande de volaille par habitant

In [136]:
# Objectif : pour chaque pays, la dispo de viande de volaille en kg/personne/an.
# Cette fois on filtre sur 2 conditions : le bon produit ET le bon élément.

# On combine deux filtres avec & (= "ET").
# Chaque condition doit être entre parenthèses.
volaille = dispo[
    (dispo["Produit"] == "Viande de Volailles")
    & (dispo["Élément"] == "Disponibilité alimentaire en quantité (kg/personne/an)")
]

# Ici, normalement une seule ligne par pays (un seul produit, un seul élément).
# On regroupe quand même par sécurité, au cas où.
volaille_par_pays = volaille.groupby("Zone")["Valeur"].sum()

# Vérification
volaille_par_pays.head()

Zone
Afghanistan        1.53
Afrique du Sud    35.69
Albanie           16.36
Algérie            6.38
Allemagne         19.47
Name: Valeur, dtype: float64

Comptage des pays présents pour chaque variable

In [137]:
# On compare le nombre de pays dans chaque variable construite jusqu'ici.
# Si la volaille a MOINS de pays que les autres, c'est que certains pays
# n'ont pas de ligne "volaille" (ils disparaîtront = valeurs manquantes à gérer).

print("Protéines  :", proteines_par_pays.shape[0], "pays")
print("Kcal       :", kcal_par_pays.shape[0], "pays")
print("Dépendance :", taux_dependance.shape[0], "pays")
print("Volaille   :", volaille_par_pays.shape[0], "pays")

Protéines  : 172 pays
Kcal       : 172 pays
Dépendance : 174 pays
Volaille   : 172 pays


Population 2017

In [138]:
# pop_2017 est déjà fait. On le revérifie juste.
print("Dimensions :", pop_2017.shape)
print("Nombre de pays :", pop_2017["Zone"].nunique())
pop_2017.head()

Dimensions : (236, 3)
Nombre de pays : 236


,Code zone,Zone,Population_2017
17,2,Afghanistan,36296.113
36,202,Afrique du Sud,57009.756
55,3,Albanie,2884.169
74,4,Algérie,41389.189
93,79,Allemagne,82658.409


Croissance démographique

In [139]:
# Objectif : mesurer l'évolution de la population entre 2000 et 2017 (en %).

# Étape 1 : extraire la population de 2000 (depuis le fichier complet "pop")
# On garde Zone + Valeur, et on renomme Valeur pour être clair.
pop_2000 = pop[pop["Année"] == 2000][["Zone", "Valeur"]]
pop_2000 = pop_2000.rename(columns={"Valeur": "Population_2000"})

# Étape 2 : on a déjà pop_2017 (Zone + Population_2017). On fusionne les deux
# sur le nom du pays pour avoir 2000 et 2017 côte à côte.

croissance = pop_2017.merge(pop_2000, on="Zone", how="inner")

# Étape 3 : on calcule le taux de croissance en %
croissance["Croissance_demo"] = (
    (croissance["Population_2017"] - croissance["Population_2000"])
    / croissance["Population_2000"] * 100
)

# Vérification
croissance[["Zone", "Population_2000", "Population_2017", "Croissance_demo"]].head()

,Zone,Population_2000,Population_2017,Croissance_demo
0,Afghanistan,20779.953,36296.113,74.668889
1,Afrique du Sud,44967.708,57009.756,26.779324
2,Albanie,3129.243,2884.169,-7.831734
3,Algérie,31042.235,41389.189,33.331859
4,Allemagne,81400.882,82658.409,1.544857


 Ratio de protéines animales

In [140]:
#Liste des produits uniques dans le fichier dispo
dispo["Produit"].unique()

array(['Blé et produits', 'Riz et produits', 'Orge et produits',
       'Maïs et produits', 'Seigle et produits', 'Avoine',
       'Millet et produits', 'Sorgho et produits', 'Céréales, Autres',
       'Pommes de Terre et produits', 'Ignames', 'Racines nda',
       'Sucre, canne', 'Sucre, betterave', 'Sucre Eq Brut',
       'Edulcorants Autres', 'Miel', 'Haricots', 'Pois',
       'Légumineuses Autres et produits', 'Noix et produits', 'Soja',
       'Arachides Decortiquees', 'Graines de tournesol',
       'Graines Colza/Moutarde', 'Graines de coton', 'Coco (Incl Coprah)',
       'Sésame', 'Olives', 'Plantes Oleiferes, Autre', 'Huile de Soja',
       "Huile d'Arachide", 'Huile de Tournesol',
       'Huile de Colza&Moutarde', 'Huile Graines de Coton',
       'Huile de Palmistes', 'Huile de Palme', 'Huile de Coco',
       'Huile de Sésame', "Huile d'Olive", 'Huile de Son de Riz',
       'Huile de Germe de Maïs', 'Huil Plantes Oleif Autr',
       'Tomates et produits', 'Oignons', 'Légumes, 

In [141]:
# Objectif : calculer la part des protéines qui vient des produits animaux.

# Étape 1 : on définit la liste des produits d'origine animale.
# (Liste choisie et justifiable : viandes, abats, laitages, œufs, poissons/fruits de mer)
produits_animaux = [
    "Viande d'Ovins/Caprins", "Viande de Anim Aquatiq", "Viande de Bovins",
    "Viande de Suides", "Viande de Volailles", "Viande, Autre", "Abats Comestible",
    "Lait - Excl Beurre", "Beurre, Ghee", "Crème", "Oeufs",
    "Poissons Eau Douce", "Poissons Marins, Autres", "Poissons Pelagiques",
    "Crustacés", "Cephalopodes", "Mollusques, Autres", "Animaux Aquatiques Autre"
]

# Étape 2 : on garde les lignes de protéines ET dont le produit est animal avec isin.
proteines_animales = dispo[
    (dispo["Élément"] == "Disponibilité de protéines en quantité (g/personne/jour)")
    & (dispo["Produit"].isin(produits_animaux))
]

# Étape 3 : on additionne par pays
proteines_animales_par_pays = proteines_animales.groupby("Zone")["Valeur"].sum()

# Étape 4 : le ratio = protéines animales / protéines totales
ratio_proteines_animales = proteines_animales_par_pays / proteines_par_pays

# Vérification
ratio_proteines_animales.head()

Zone
Afghanistan       0.195045
Afrique du Sud    0.408709
Albanie           0.552050
Algérie           0.275821
Allemagne         0.588354
Name: Valeur, dtype: float64

Convertir les Series en DataFrames

In [142]:
# Les Series ont "Zone" comme index. On veut "Zone" comme vraie colonne.
# .reset_index() transforme l'index en colonne.
# .rename(...) donne un nom clair à la colonne de valeurs.

proteines_df = proteines_par_pays.reset_index().rename(
    columns={"Valeur": "Proteines_totales"})

kcal_df = kcal_par_pays.reset_index().rename(
    columns={"Valeur": "Kcal_totales"})

volaille_df = volaille_par_pays.reset_index().rename(
    columns={"Valeur": "Volaille_kg_hab"})

dependance_df = taux_dependance.reset_index().rename(
    columns={"Valeur": "Taux_dependance_import"})

# Pour le ratio, la Series n'a pas de nom de colonne "Valeur" c'est un calcul fait.
# On la convertit et on nomme la colonne directement.
ratio_df = ratio_proteines_animales.reset_index()
ratio_df.columns = ["Zone", "Ratio_proteines_animales"]

# Vérification : on regarde un des résultats
proteines_df.head()

,Zone,Proteines_totales
0,Afghanistan,54.09
1,Afrique du Sud,83.36
2,Albanie,119.50
3,Algérie,92.85
4,Allemagne,104.07


In [143]:
# On part de "croissance" qui contient déjà population 2000, 2017 et croissance.
# On ne garde que les colonnes utiles pour la suite.
fao = croissance[["Zone", "Population_2017", "Croissance_demo"]]

# On fusionne les autres variables FAO une par une, sur la colonne "Zone".
# how="outer" = on garde tous les pays, même ceux présents dans un seul tableau.
fao = fao.merge(proteines_df,  on="Zone", how="outer")
fao = fao.merge(kcal_df,       on="Zone", how="outer")
fao = fao.merge(volaille_df,   on="Zone", how="outer")
fao = fao.merge(dependance_df, on="Zone", how="outer")
fao = fao.merge(ratio_df,      on="Zone", how="outer")

# Vérification : dimensions et aperçu
print("Dimensions du bloc FAO :", fao.shape)
print("Nombre de pays :", fao["Zone"].nunique())
fao.head()

Dimensions du bloc FAO : (230, 8)
Nombre de pays : 230


,Zone,Population_2017,Croissance_demo,Proteines_totales,Kcal_totales,Volaille_kg_hab,Taux_dependance_import,Ratio_proteines_animales
0,Afghanistan,36296.113,74.668889,54.09,1997.0,1.53,0.300879,0.195045
1,Afrique du Sud,57009.756,26.779324,83.36,2987.0,35.69,0.134949,0.408709
2,Albanie,2884.169,-7.831734,119.50,3400.0,16.36,0.181185,0.552050
3,Algérie,41389.189,33.331859,92.85,3345.0,6.38,0.434652,0.275821
4,Allemagne,82658.409,1.544857,104.07,3559.0,19.47,0.380373,0.588354


<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="nettoyage"><h2 style="margin: auto; padding: 20px; color:#fff;">4. Nettoyage du bloc FAO et valeurs manquantes</h2></a>
</div>


In [144]:
# Combien de valeurs manquantes par colonne dans le bloc FAO ?
print("=== Valeurs manquantes par colonne ===")
print(fao.isnull().sum())

print()
print("Nombre total de pays dans le bloc FAO :", fao.shape[0])

# On affiche les pays qui ont AU MOINS un trou, pour les voir concrètement.
print()
print("=== Pays avec au moins une valeur manquante ===")
print(fao[fao.isnull().any(axis=1)])

=== Valeurs manquantes par colonne ===
Zone                         0
Population_2017              3
Croissance_demo              3
Proteines_totales           58
Kcal_totales                58
Volaille_kg_hab             58
Taux_dependance_import      56
Ratio_proteines_animales    58
dtype: int64

Nombre total de pays dans le bloc FAO : 230

=== Pays avec au moins une valeur manquante ===
                            Zone  Population_2017  Croissance_demo  \
5                        Andorre           77.001        17.756538   
7                       Anguilla           14.584        29.612513   
9    Antilles néerlandaises (ex)          275.186        27.720819   
13                         Aruba          105.366        15.974156   
18                       Bahreïn         1494.076       124.804585   
..                           ...              ...              ...   
224                Îles Marshall           58.058        14.393238   
226      Îles Turques-et-Caïques           37.

In [145]:
# On garde les pays qui ONT une donnée alimentaire.
# Si "Proteines_totales" est renseignée, le pays a des données FAO alimentaires.
# On supprime les lignes où cette colonne est vide.
fao = fao.dropna(subset=["Proteines_totales"])

# Vérification
print("Nombre de pays après nettoyage :", fao.shape[0])
print()
print("=== Valeurs manquantes restantes par colonne ===")
print(fao.isnull().sum())

Nombre de pays après nettoyage : 172

=== Valeurs manquantes restantes par colonne ===
Zone                        0
Population_2017             3
Croissance_demo             3
Proteines_totales           0
Kcal_totales                0
Volaille_kg_hab             0
Taux_dependance_import      0
Ratio_proteines_animales    0
dtype: int64


In [146]:
# On affiche les pays qui ont un trou sur la population.
# Ils ont des données alimentaires mais pas de population : pourquoi ?
fao[fao["Population_2017"].isnull()]

,Zone,Population_2017,Croissance_demo,Proteines_totales,Kcal_totales,Volaille_kg_hab,Taux_dependance_import,Ratio_proteines_animales
123,Monténégro,NaN,NaN,113.12,3478.0,15.98,0.616322,0.600778
177,Serbie,NaN,NaN,82.43,2799.0,10.16,0.084384,0.482106
184,Soudan,NaN,NaN,67.88,2431.0,1.60,0.129448,0.311137


In [147]:
# On retire les pays dont la croissance démographique est manquante.
# Monténégro, Serbie, Soudan : ruptures historiques 2000-2017 empêchant
#  un calcul de croissance cohérent.
fao = fao.dropna(subset=["Croissance_demo"])

# Vérification : plus aucun trou nulle part ?
print("Nombre de pays :", fao.shape[0])
print()
print("=== Valeurs manquantes par colonne ===")
print(fao.isnull().sum())

Nombre de pays : 169

=== Valeurs manquantes par colonne ===
Zone                        0
Population_2017             0
Croissance_demo             0
Proteines_totales           0
Kcal_totales                0
Volaille_kg_hab             0
Taux_dependance_import      0
Ratio_proteines_animales    0
dtype: int64


<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="fusion"><h2 style="margin: auto; padding: 20px; color:#fff;">5. Fusion des sources (PIB et stabilité politique)</h2></a>
</div>


In [148]:
import country_converter as coco
cc = coco.CountryConverter()

# --- Étape 1 : récupérer la correspondance Code zone & Zone ---
# dispo contient encore Code zone + Zone. Chaque pays y apparaît des milliers
# de fois (format long), donc on enlève les doublons pour avoir 1 ligne par pays.
correspondance = dispo[["Code zone", "Zone"]].drop_duplicates()

# --- Étape 2 : ajouter Code zone au bloc fao, en reliant par le nom Zone ---
# Zone & Zone, même fichier FAO français : correspondance parfaite.
fao = fao.merge(correspondance, on="Zone", how="left")

# --- Étape 3 : convertir le Code zone (FAOcode) en ISO3 ---
fao["ISO3"] = cc.convert(fao["Code zone"], src="FAOcode", to="ISO3", not_found=None)

# --- Vérification : des pays sans code ISO ? ---
print("Pays sans code ISO trouvé :")
print(fao[fao["ISO3"].isnull()][["Zone", "Code zone", "ISO3"]])
print()
print("Total pays :", fao.shape[0], "| Codes ISO trouvés :", fao["ISO3"].notnull().sum())

41 not found in FAOcode


Pays sans code ISO trouvé :
Empty DataFrame
Columns: [Zone, Code zone, ISO3]
Index: []

Total pays : 169 | Codes ISO trouvés : 169


In [149]:
# --- PIB : on recharge en gardant le Country Code (= ISO3) ---
pib = pd.read_csv("API_NY.GDP.PCAP.CD_DS2_en_csv_v2_465995.csv", skiprows=4)
# Cette fois on garde Country Code en plus du nom et de 2017
pib = pib[["Country Code", "2017"]]
pib = pib.rename(columns={"Country Code": "ISO3", "2017": "PIB_par_hab_2017"})

# --- Stabilité : le fichier DataBank a aussi un Country Code (= ISO3) ---
stab = pd.read_csv("f09c2b69-99f0-4702-ad1f-43a99cb25d78_Data.csv")
stab = stab[["Country Code", "2017 [YR2017]"]]
stab = stab.rename(columns={"Country Code": "ISO3", "2017 [YR2017]": "Stabilite_politique_2017"})
# On reconvertit en nombre (le blabla de fin devient NaN) et on enlève les NaN
stab["Stabilite_politique_2017"] = pd.to_numeric(stab["Stabilite_politique_2017"], errors="coerce")
stab = stab.dropna(subset=["Stabilite_politique_2017"])

# Vérification
print("PIB :", pib.shape, "| colonnes :", pib.columns.tolist())
print("Stabilité :", stab.shape, "| colonnes :", stab.columns.tolist())
print()
print("Aperçu PIB :")
print(pib.head(3))
print()
print("Aperçu stabilité :")
print(stab.head(3))

PIB : (266, 2) | colonnes : ['ISO3', 'PIB_par_hab_2017']
Stabilité : (212, 2) | colonnes : ['ISO3', 'Stabilite_politique_2017']

Aperçu PIB :
  ISO3  PIB_par_hab_2017
0  ABW      28440.041688
1  AFE       1528.104224
2  AFG        525.469771

Aperçu stabilité :
  ISO3  Stabilite_politique_2017
0  AFG                 -2.613931
1  ALB                  0.163830
2  DZA                 -0.929505


In [150]:
# On part du bloc FAO qui est notre référence : 169 pays bien définis.
# On ajoute PIB puis stabilité, en reliant sur le code ISO3.
# how="left" = on garde TOUS les pays du bloc FAO, et on rattache les infos
#              Banque mondiale quand le code ISO correspond.
#              Si un pays FAO n'a pas de PIB, sa case PIB sera NaN.
df = fao.merge(pib,  on="ISO3", how="left")
df = df.merge(stab, on="ISO3", how="left")

# Vérification : dimensions et valeurs manquantes
print("Dimensions du tableau final :", df.shape)
print("Nombre de pays :", df.shape[0])
print()
print("=== Valeurs manquantes par colonne ===")
print(df.isnull().sum())

Dimensions du tableau final : (169, 12)
Nombre de pays : 169

=== Valeurs manquantes par colonne ===
Zone                        0
Population_2017             0
Croissance_demo             0
Proteines_totales           0
Kcal_totales                0
Volaille_kg_hab             0
Taux_dependance_import      0
Ratio_proteines_animales    0
Code zone                   0
ISO3                        0
PIB_par_hab_2017            3
Stabilite_politique_2017    3
dtype: int64


In [151]:
# Pays avec un trou sur le PIB
print("=== Pays sans PIB ===")
print(df[df["PIB_par_hab_2017"].isnull()][["Zone", "ISO3", "PIB_par_hab_2017"]])

print()
# Pays avec un trou sur la stabilité
print("=== Pays sans stabilité politique ===")
print(df[df["Stabilite_politique_2017"].isnull()][["Zone", "ISO3", "Stabilite_politique_2017"]])

=== Pays sans PIB ===
                                           Zone ISO3  PIB_par_hab_2017
33                    Chine, Taiwan Province de  TWN               NaN
34                          Chine, continentale   41               NaN
130  République populaire démocratique de Corée  PRK               NaN

=== Pays sans stabilité politique ===
                    Zone ISO3  Stabilite_politique_2017
34   Chine, continentale   41                       NaN
107   Nouvelle-Calédonie  NCL                       NaN
119  Polynésie française  PYF                       NaN


In [152]:
# La "Chine, continentale" a gardé le code FAO brut "41" au lieu de "CHN".
# On corrige ça directement dans le tableau.
df.loc[df["Zone"] == "Chine, continentale", "ISO3"] = "CHN"

# Vérification : la Chine a-t-elle bien CHN maintenant ?
print(df[df["Zone"] == "Chine, continentale"][["Zone", "ISO3"]])

                   Zone ISO3
34  Chine, continentale  CHN


In [153]:
# La Chine a maintenant le bon ISO3 (CHN), mais ses cases PIB et stabilité
# sont encore vides car la fusion a eu lieu avant la correction.
# On va chercher les valeurs CHN dans les blocs pib et stab, et les injecter.

# --- Récupérer la valeur PIB de la Chine (code CHN) ---
pib_chine = pib[pib["ISO3"] == "CHN"]["PIB_par_hab_2017"].values[0]

# --- Récupérer la valeur stabilité de la Chine (code CHN) ---
stab_chine = stab[stab["ISO3"] == "CHN"]["Stabilite_politique_2017"].values[0]

# --- Injecter ces valeurs dans la ligne Chine du tableau final ---
df.loc[df["Zone"] == "Chine, continentale", "PIB_par_hab_2017"] = pib_chine
df.loc[df["Zone"] == "Chine, continentale", "Stabilite_politique_2017"] = stab_chine

# Vérification
print("Valeurs récupérées pour la Chine :")
print("  PIB/hab :", pib_chine)
print("  Stabilité :", stab_chine)
print()
print(df[df["Zone"] == "Chine, continentale"][
    ["Zone", "ISO3", "PIB_par_hab_2017", "Stabilite_politique_2017"]])

Valeurs récupérées pour la Chine :
  PIB/hab : 8979.67652709856
  Stabilité : -0.203288

                   Zone ISO3  PIB_par_hab_2017  Stabilite_politique_2017
34  Chine, continentale  CHN       8979.676527                 -0.203288


In [154]:
# On supprime toutes les lignes qui ont encore un trou (PIB ou stabilité).
# On supprime toute ligne ayant au moins un NaN, n'importe où.
df = df.dropna()

# Vérification finale
print("Nombre de pays final :", df.shape[0])
print()
print("=== Valeurs manquantes par colonne (doit être 0 partout) ===")
print(df.isnull().sum())

Nombre de pays final : 165

=== Valeurs manquantes par colonne (doit être 0 partout) ===
Zone                        0
Population_2017             0
Croissance_demo             0
Proteines_totales           0
Kcal_totales                0
Volaille_kg_hab             0
Taux_dependance_import      0
Ratio_proteines_animales    0
Code zone                   0
ISO3                        0
PIB_par_hab_2017            0
Stabilite_politique_2017    0
dtype: int64


<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="couverture"><h2 style="margin: auto; padding: 20px; color:#fff;">6. Couverture de la population mondiale</h2></a>
</div>


In [155]:
# Population totale couverte par nos 165 pays.
# Rappel : la population est en "milliers de personnes" (1000 personnes).
pop_couverte = df["Population_2017"].sum()

# Population mondiale 2017 ≈ 7,55 milliards = 7 550 000 milliers
pop_mondiale_2017 = 7_550_000  # en milliers de personnes

# Pourcentage de couverture
couverture = pop_couverte / pop_mondiale_2017 * 100

print("Population couverte (milliers) :", round(pop_couverte))
print("Population couverte (milliards) :", round(pop_couverte / 1_000_000, 2))
print("Couverture de la population mondiale : {:.1f} %".format(couverture))

Population couverte (milliers) : 7268854
Population couverte (milliards) : 7.27
Couverture de la population mondiale : 96.3 %


<div style="background-color: RGB(0,150,250);">
<a class="anchor" id="save"><h2 style="margin: auto; padding: 20px; color:#fff;">7. Enregistrement du jeu de données</h2></a>
</div>


In [156]:
# On sauvegarde le tableau final propre en CSV.
df.to_csv("donnees_pretes.csv", index=False)

print("Fichier sauvegardé : donnees_pretes.csv")
print("Dimensions :", df.shape)
print()
# Petit aperçu final de tout le tableau
df.head()

Fichier sauvegardé : donnees_pretes.csv
Dimensions : (165, 12)



,Zone,Population_2017,Croissance_demo,Proteines_totales,Kcal_totales,Volaille_kg_hab,Taux_dependance_import,Ratio_proteines_animales,Code zone,ISO3,PIB_par_hab_2017,Stabilite_politique_2017
0,Afghanistan,36296.113,74.668889,54.09,1997.0,1.53,0.300879,0.195045,2,AFG,525.469771,-2.613931
1,Afrique du Sud,57009.756,26.779324,83.36,2987.0,35.69,0.134949,0.408709,202,ZAF,6618.335083,-0.382832
2,Albanie,2884.169,-7.831734,119.50,3400.0,16.36,0.181185,0.552050,3,ALB,5006.360130,0.163830
3,Algérie,41389.189,33.331859,92.85,3345.0,6.38,0.434652,0.275821,4,DZA,4554.667540,-0.929505
4,Allemagne,82658.409,1.544857,104.07,3559.0,19.47,0.380373,0.588354,79,DEU,45553.934150,0.702224
